# Export Fine-Tuned Model to GGUF (for Ollama)

**Project:** Aspect-Level Customer Review Intelligence via Fine-Tuned Small Language Models

Merges the LoRA adapter (already trained in `03_finetune_colab.ipynb`) into
the base model and exports a Q4_K_M-quantized GGUF file, so the fine-tuned
model can be served locally through Ollama for Stage 8 (the application
layer over the earphone review corpus) — same serving mechanism as the
zero-shot baseline, for a fair apples-to-apples comparison of quantization
level.

**This does NOT redo fine-tuning.** It reloads the adapter you already have
in the local repo's `adapters/final/` and just merges + exports it. Should
take a few minutes, not the ~30-60 min the original training run took.

**Runtime:** Colab, T4 GPU (Runtime → Change runtime type → T4 GPU) — the
merge step benefits from the same GPU the adapter was trained on.

**Revision history:** the first version of this notebook loaded the base
model and attached the adapter via plain `peft.PeftModel.from_pretrained()`.
That actually ran and failed with `RuntimeError: Failed to save model:
NotImplementedError` at the export step — Unsloth's `save_pretrained_gguf`
checks for an Unsloth-specific "is this a PEFT model" marker that a
vanilla-`peft`-wrapped model doesn't carry, so it silently skipped the LoRA
merge and tried to serialize the still-4-bit-quantized base directly, which
isn't supported. Fixed in Section 3 below by loading the adapter through
`FastLanguageModel.from_pretrained` directly (Unsloth's own adapter-aware
path) instead. The rest of the pipeline shape (load → merge → export →
quantize) is unchanged and still not GPU-tested end to end by me.

**Correctness detail that matters more than any remaining library-version
risk:** the fine-tuned model was trained on **raw** concatenated text —
never wrapped in Qwen's chat template. The Ollama Modelfile built from this
GGUF (prepared locally, not here) uses an explicit passthrough template so
Ollama doesn't wrap prompts in a chat format the model never saw during
training. That's handled after this notebook, once the `.gguf` file is
downloaded.


## 1. Setup

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes


In [ ]:
from pathlib import Path
from unsloth import FastLanguageModel

WORKDIR = Path("/content/asmta")
ADAPTER_DIR = WORKDIR / "adapters" / "final"
GGUF_OUT_DIR = WORKDIR / "gguf_export"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
GGUF_OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 512  # same as training


## 2. Upload the already-trained adapter (local, not Drive)

Select all files from the local repo's `adapters/final/` at once:
`adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`,
`tokenizer_config.json`, `chat_template.jinja` (the last one is not used —
the export below deliberately bypasses it, per the raw-text training note
above).


In [ ]:
from google.colab import files as colab_files
import shutil

print("Select all files from adapters/final/ (local repo):")
uploaded = colab_files.upload()

required = {"adapter_config.json", "adapter_model.safetensors"}
missing = required - set(uploaded.keys())
if missing:
    raise RuntimeError(f"Missing required adapter file(s): {missing}")

for fname in uploaded:
    shutil.move(fname, ADAPTER_DIR / fname)

print("Uploaded adapter files to", ADAPTER_DIR)


## 3. Load the adapter (via Unsloth's own path, not plain peft.PeftModel)

**Point `FastLanguageModel.from_pretrained` directly at the saved adapter
directory** — don't load the base separately and wrap it with
`peft.PeftModel.from_pretrained()`. Unsloth auto-detects an adapter
directory (via `adapter_config.json`), resolves `base_model_name_or_path`
from it (`unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit`, already confirmed present
in the saved adapter's own config) and loads+attaches everything through its
own patched code path. That matters because `save_pretrained_gguf` (used
below) is an Unsloth method that checks for an Unsloth-specific "this is a
PEFT model" marker to decide whether to merge before saving — a model
wrapped with the plain `peft` library's `PeftModel` doesn't carry that
marker, so Unsloth's check fails, it silently skips the merge, and tries to
serialize the still-4-bit-quantized base directly — which isn't supported
and throws `NotImplementedError`. This was tried and hit exactly that error;
loading through `FastLanguageModel.from_pretrained` avoids it.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),  # the saved adapter dir, NOT the base model name
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print("Base model resolved from adapter_config.json and adapter attached "
      "(both via Unsloth's own loading path).")


## 4. Merge and export to GGUF (Q4_K_M)

`quantization_method="q4_k_m"` matches the quantization level Ollama
already uses for the zero-shot baseline model (`qwen2.5:1.5b-instruct` is
Q4_K_M) — keeping quantization level constant means the eventual
zero-shot-vs-fine-tuned application-layer comparison isn't confounded by
"maybe it's just less quantized now."


In [ ]:
model.save_pretrained_gguf(
    str(GGUF_OUT_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = list(GGUF_OUT_DIR.glob("*.gguf"))
print("GGUF file(s) produced:", gguf_files)


## 5. Download the GGUF file

In [ ]:
assert gguf_files, "No .gguf file found — check the export cell above for errors."
gguf_path = gguf_files[0]
print(f"Downloading {gguf_path} ({gguf_path.stat().st_size / 1e6:.0f} MB)")
colab_files.download(str(gguf_path))


### Bring this back to the local repo

Save the downloaded `.gguf` file into the local repo at, e.g.,
`adapters/gguf/qwen2.5-1.5b-absa-finetuned.q4_k_m.gguf`.

**Local steps from here need no GPU and no Colab** — creating the Ollama
Modelfile (with the required passthrough template) and running
`ollama create` happens locally, using the Ollama instance already running
on this machine.
